In [1]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [2]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql", "sql/01_clean.sql", "sql/02_features.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

feat = load_queries("sql/02_features.sql")
display(run(feat, "features_reconcile").T)

,0
n_rows,20405666
pu_zone_null,0
pu_borough_null,0
min_hour,0
max_hour,23
min_isodow,1
max_isodow,7
n_pickup_zones,262
min_request_date,2024-12-31 00:00:00
max_request_date,2025-02-01 00:00:00


In [3]:
e = load_queries("sql/02e_company_like_for_like.sql")
for name in ["create_ce_bins", "create_ce_strata", "create_ce_weights", "create_ce_std_pdf", "create_ce_stratum_stats"]:
    print("building", name)
    con.execute(e[name])

building create_ce_bins
building create_ce_strata
building create_ce_weights
building create_ce_std_pdf
building create_ce_stratum_stats


In [4]:
display(run(e, "ce_reconcile"))
display(run(e, "ce_design_c_crosscheck").T)
display(run(e, "ce_binned_method_check"))
display(run(e, "ce_weights_check"))
display(run(e, "ce_pdf_check"))
display(run(e, "ce_identity_check"))

,company,n_scoped,n_valid,n_excluded,n_wait_ge10
0,Uber,15355926.0,15234971.0,120955.0,814968.0
1,Lyft,5048966.0,4970147.0,78819.0,236554.0
2,All,20404892.0,20205118.0,199774.0,1051522.0


,0
n_strata_observed,2579.00
n_strata_with_valid,2579.00
n_pooled_adequate,2177.00
pct_valid_in_pooled_adequate,99.18
n_strata_missing_a_company,17.00


,company,n_valid,p50,p75,p90,p95,p99,share_ge10_pct
0,Uber,15234971.0,3.807,5.522,8.050,10.221,15.795,5.349
1,Lyft,4970147.0,4.087,5.773,8.041,9.868,14.545,4.759


,n_combos,min_weight_sum,max_weight_sum,min_strata_in_a_combo
0,55,1.0,1.0,15


,n_combo_company,min_pdf_sum,max_pdf_sum
0,110,1.0,1.0


,min_n,company,direct_share,std_share,abs_diff
0,1000,Lyft,0.046177,0.046177,1.942890e-16
1,1000,Uber,0.051446,0.051446,1.387779e-17
2,500,Lyft,0.046464,0.046464,4.649059e-16
3,500,Uber,0.052065,0.052065,3.122502e-16
4,300,Lyft,0.046879,0.046879,8.326673e-17
5,300,Uber,0.052561,0.052561,1.387779e-16
6,200,Lyft,0.047112,0.047112,1.110223e-16
7,200,Uber,0.052842,0.052842,1.040834e-16
8,100,Lyft,0.047417,0.047417,1.595946e-16
9,100,Uber,0.053252,0.053252,9.714451e-17


In [5]:
sc = run(e, "ce_support_curve"); display(sc)
sb = run(e, "ce_support_by_borough")
display(sb.pivot(index="min_n", columns="pu_borough", values="lyft_pct_in_support").sort_index(ascending=False))
display(sb.pivot(index="min_n", columns="pu_borough", values="uber_pct_in_support").sort_index(ascending=False))

checks = []
for T in sorted(sc["min_n"], reverse=True):
    r = sc[sc["min_n"] == T].iloc[0]
    b = sb[sb["min_n"] == T]
    ok = (r["uber_pct_in_support"] >= 90 and r["lyft_pct_in_support"] >= 90
          and (b["uber_pct_in_support"] >= 50).all() and (b["lyft_pct_in_support"] >= 50).all())
    checks.append((int(T), bool(ok)))
print(checks)
PRIMARY_T = next((T for T, ok in checks if ok), None)
print("PRIMARY minimum valid waits per company per stratum:", PRIMARY_T)

,min_n,n_strata_with_valid,n_strata_common_support,uber_valid_total,uber_valid_in_support,uber_pct_in_support,lyft_valid_total,lyft_valid_in_support,lyft_pct_in_support
0,1000,2579,1353,15234971.0,13426131.0,88.13,4970147.0,4460318.0,89.74
1,500,2579,1833,15234971.0,14648198.0,96.15,4970147.0,4810871.0,96.80
2,300,2579,2075,15234971.0,15004395.0,98.49,4970147.0,4907072.0,98.73
3,200,2579,2198,15234971.0,15117121.0,99.23,4970147.0,4937693.0,99.35
4,100,2579,2343,15234971.0,15198326.0,99.76,4970147.0,4959687.0,99.79


pu_borough,Bronx,Brooklyn,Manhattan,Queens,Staten Island
min_n,,,,,
1000,76.45,91.49,96.36,84.20,33.23
500,94.01,97.77,99.25,94.24,60.28
300,98.57,99.23,99.71,97.70,74.84
200,99.12,99.75,99.82,98.83,85.72
100,99.65,99.97,99.94,99.56,96.05


pu_borough,Bronx,Brooklyn,Manhattan,Queens,Staten Island
min_n,,,,,
1000,75.34,90.63,96.16,83.40,29.48
500,93.64,97.50,99.12,93.83,55.69
300,98.46,99.14,99.65,97.65,71.24
200,99.03,99.74,99.79,98.91,83.11
100,99.59,99.96,99.92,99.69,94.91


[(1000, False), (500, True), (300, True), (200, True), (100, True)]
PRIMARY minimum valid waits per company per stratum: 500


In [6]:
ss = run(e, "ce_standardized_summary")
cols = ["uber_p50","lyft_p50","diff_p50_sec","uber_p75","lyft_p75","diff_p75_sec","uber_p90","lyft_p90","diff_p90_sec",
        "uber_p95","lyft_p95","diff_p95_sec","uber_p99","lyft_p99","diff_p99_sec",
        "uber_share_ge10_pct","lyft_share_ge10_pct","diff_share_pts"]
print("PRIMARY: supported strata, pooled weights")
display(ss[(ss.grp == "ALL") & (ss.scheme == "pooled") & (ss.min_n == PRIMARY_T)][["min_n", "grp", "scheme"] + cols].T)
print("Robustness 1: minimum-sample ladder x weighting schemes")
display(ss[ss.grp == "ALL"].sort_values(["scheme", "min_n"], ascending=[True, False])[["min_n", "scheme"] + cols])

PRIMARY: supported strata, pooled weights


,16
min_n,500
grp,ALL
scheme,pooled
uber_p50,3.78
lyft_p50,4.086
diff_p50_sec,18.4
uber_p75,5.481
lyft_p75,5.771
diff_p75_sec,17.4
uber_p90,7.979


Robustness 1: minimum-sample ladder x weighting schemes


,min_n,scheme,uber_p50,lyft_p50,diff_p50_sec,uber_p75,lyft_p75,diff_p75_sec,uber_p90,lyft_p90,diff_p90_sec,uber_p95,lyft_p95,diff_p95_sec,uber_p99,lyft_p99,diff_p99_sec,uber_share_ge10_pct,lyft_share_ge10_pct,diff_share_pts
5,1000,equal,3.796,4.109,18.8,5.451,5.735,17.0,7.875,7.902,1.7,9.980,9.645,-20.1,15.404,14.077,-79.6,4.968,4.361,-0.607
6,500,equal,3.863,4.177,18.8,5.525,5.819,17.6,7.984,8.003,1.1,10.146,9.761,-23.1,15.646,14.280,-82.0,5.226,4.562,-0.664
7,300,equal,3.907,4.224,19.0,5.590,5.892,18.1,8.089,8.119,1.8,10.295,9.909,-23.1,15.852,14.551,-78.1,5.461,4.827,-0.634
8,200,equal,3.936,4.256,19.2,5.640,5.944,18.2,8.174,8.203,1.8,10.413,10.019,-23.6,15.997,14.756,-74.5,5.648,5.035,-0.613
9,100,equal,3.983,4.304,19.3,5.734,6.035,18.1,8.353,8.368,0.9,10.673,10.248,-25.5,16.374,15.221,-69.2,6.062,5.464,-0.598
10,1000,lyft_mix,3.744,4.044,18.0,5.448,5.718,16.2,7.931,7.971,2.4,10.041,9.788,-15.2,15.518,14.393,-67.5,5.064,4.618,-0.446
11,500,lyft_mix,3.766,4.067,18.1,5.468,5.742,16.4,7.955,7.991,2.2,10.079,9.804,-16.5,15.574,14.419,-69.3,5.123,4.646,-0.476
12,300,lyft_mix,3.775,4.077,18.1,5.480,5.755,16.5,7.972,8.011,2.4,10.104,9.828,-16.6,15.610,14.461,-68.9,5.162,4.688,-0.474
13,200,lyft_mix,3.779,4.081,18.1,5.486,5.762,16.6,7.982,8.022,2.4,10.118,9.841,-16.6,15.630,14.485,-68.6,5.185,4.711,-0.473
14,100,lyft_mix,3.783,4.085,18.1,5.493,5.769,16.6,7.994,8.035,2.4,10.137,9.858,-16.7,15.661,14.521,-68.4,5.215,4.742,-0.473


In [7]:
print("Robustness 2: airports (pooled weights, primary min_n)")
display(ss[ss.grp.isin(["ALL", "EX_AIRPORTS", "AIRPORTS"]) & (ss.scheme == "pooled") & (ss.min_n == PRIMARY_T)][["grp"] + cols])
print("By borough (pooled weights, primary min_n)")
display(ss[ss.grp.str.startswith("BOROUGH") & (ss.min_n == PRIMARY_T)][["grp"] + cols])
display(run(e, "ce_standardized_support"))
print("Exclusion bounds on the share of waits >= 10 min")
display(run(e, "ce_share_exclusion_bounds"))

Robustness 2: airports (pooled weights, primary min_n)


,grp,uber_p50,lyft_p50,diff_p50_sec,uber_p75,lyft_p75,diff_p75_sec,uber_p90,lyft_p90,diff_p90_sec,uber_p95,lyft_p95,diff_p95_sec,uber_p99,lyft_p99,diff_p99_sec,uber_share_ge10_pct,lyft_share_ge10_pct,diff_share_pts
1,AIRPORTS,6.310,4.368,-116.5,9.152,8.765,-23.2,12.480,12.204,-16.6,15.559,15.046,-30.8,27.295,22.602,-281.6,19.751,18.337,-1.414
16,ALL,3.780,4.086,18.4,5.481,5.771,17.4,7.979,8.041,3.7,10.118,9.867,-15.1,15.635,14.513,-67.3,5.186,4.757,-0.429
51,EX_AIRPORTS,3.731,4.083,21.2,5.354,5.708,21.2,7.702,7.854,9.1,9.754,9.574,-10.8,15.089,13.921,-70.1,4.621,4.230,-0.391


By borough (pooled weights, primary min_n)


,grp,uber_p50,lyft_p50,diff_p50_sec,uber_p75,lyft_p75,diff_p75_sec,uber_p90,lyft_p90,diff_p90_sec,uber_p95,lyft_p95,diff_p95_sec,uber_p99,lyft_p99,diff_p99_sec,uber_share_ge10_pct,lyft_share_ge10_pct,diff_share_pts
26,BOROUGH:Bronx,4.010,4.469,27.5,5.648,6.220,34.3,8.102,8.536,26.1,10.245,10.322,4.6,15.589,14.780,-48.5,5.393,5.660,0.267
31,BOROUGH:Brooklyn,3.888,4.161,16.4,5.521,5.780,15.5,7.916,7.936,1.2,10.017,9.657,-21.6,15.447,13.989,-87.5,5.026,4.364,-0.663
36,BOROUGH:Manhattan,3.445,3.829,23.0,5.018,5.372,21.3,7.213,7.356,8.6,9.096,8.967,-7.7,14.165,13.118,-62.8,3.675,3.273,-0.402
41,BOROUGH:Queens,4.131,4.244,6.8,6.201,6.270,4.1,9.225,9.088,-8.2,11.602,11.219,-23.0,17.957,17.012,-56.7,7.943,7.437,-0.506
46,BOROUGH:Staten Island,4.790,4.873,5.0,7.021,6.894,-7.6,10.251,9.483,-46.1,12.893,11.485,-84.5,18.750,16.403,-140.9,10.679,8.265,-2.415


,min_n,grp,n_strata,uber_valid_in_support,uber_pct_of_group,lyft_valid_in_support,lyft_pct_of_group,max_stratum_weight_pct,airport_weight_pct
0,1000,AIRPORTS,18,558918.0,98.95,160020.0,98.84,12.93,100.00
1,500,AIRPORTS,20,564829.0,100.00,161895.0,100.00,12.79,100.00
2,300,AIRPORTS,20,564829.0,100.00,161895.0,100.00,12.79,100.00
3,200,AIRPORTS,20,564829.0,100.00,161895.0,100.00,12.79,100.00
4,100,AIRPORTS,20,564829.0,100.00,161895.0,100.00,12.79,100.00
5,1000,ALL,1353,13426131.0,88.13,4460318.0,89.74,0.52,4.02
6,500,ALL,1833,14648198.0,96.15,4810871.0,96.80,0.48,3.73
7,300,ALL,2075,15004395.0,98.49,4907072.0,98.73,0.47,3.65
8,200,ALL,2198,15117121.0,99.23,4937693.0,99.35,0.46,3.62
9,100,ALL,2343,15198326.0,99.76,4959687.0,99.79,0.46,3.61


Exclusion bounds on the share of waits >= 10 min


,min_n,uber_share_valid_pct,lyft_share_valid_pct,diff_share_valid_pts,uber_share_if_excluded_short_pct,uber_share_if_excluded_long_pct,lyft_share_if_excluded_short_pct,lyft_share_if_excluded_long_pct,diff_lower_bound_pts,diff_upper_bound_pts
0,1000,5.124,4.723,-0.401,5.082,5.867,4.636,6.181,-1.231,1.099
1,500,5.186,4.757,-0.429,5.143,5.925,4.671,6.191,-1.253,1.049
2,300,5.233,4.807,-0.426,5.189,5.972,4.721,6.236,-1.251,1.047
3,200,5.260,4.835,-0.425,5.215,5.999,4.748,6.264,-1.251,1.048
4,100,5.298,4.874,-0.424,5.253,6.039,4.785,6.303,-1.254,1.050


dsum = run(e, "ce_stratum_diff_summary"); display(dsum)
display(run(e, "ce_stratum_extremes"))
cmp_ = run(e, "ce_stratum_compare"); print(len(cmp_), "strata with >= 100 valid waits per company")

In [9]:
out = Path("dashboard/data"); out.mkdir(parents=True, exist_ok=True)
sc.to_csv(out / "2e_support_curve.csv", index=False)
sb.to_csv(out / "2e_support_by_borough.csv", index=False)
ss.to_csv(out / "2e_standardized_summary.csv", index=False)
run(e, "ce_standardized_support").to_csv(out / "2e_standardized_support.csv", index=False)
run(e, "ce_share_exclusion_bounds").to_csv(out / "2e_share_exclusion_bounds.csv", index=False)
dsum.to_csv(out / "2e_stratum_diff_summary.csv", index=False)
cmp_.to_csv(out / "2e_stratum_compare.csv", index=False)